# Day 1 exercise: Email summarizer

Hands-on variation of the Week 1 website summarizer.

Write a **system prompt** and a **user prompt**, then call the OpenAI Chat Completions API to:

1. Summarize an email
2. Suggest a short, scannable subject line

Select the repo `.venv` kernel (same as `week1/day1.ipynb`) before running.

The JS-site scraper for openai.com is in `openai_summarizer.ipynb`.

In [ ]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

## Connect to OpenAI

Load `OPENAI_API_KEY` from the project `.env` (repo root). If this cell fails, use the troubleshooting notebook in `setup/`.

In [ ]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key was found — check your .env and the troubleshooting notebook.")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key.")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end.")
else:
    print("API key found and looks good so far!")

openai = OpenAI()

## The email to summarize

Paste any email into `email` and re-run the cells below.

In [ ]:
email = """
From: Manish Kumar <manish@acme.com>
To: Hitesh Kumar <hitesh@acme.com>
Date: Mon, 16 Aug 2026 10:12:00 +0530

Hi Hitesh,

Quick update on the Q3 launch. Engineering is still waiting on legal review of the
privacy copy, so we cannot ship the onboarding flow on Thursday as planned.

Can you:
1. Confirm whether we can use last quarter's privacy wording as a stopgap
2. Join a 20-minute call tomorrow at 3pm IST with Legal and Product
3. Send a one-paragraph status to leadership by EOD Tuesday

If we miss Thursday, the fallback is next Monday. Please flag if that's a problem
for the marketing campaign.

Thanks,
Priya
"""

## Types of prompts

- **System prompt** — who the model is, and how it should format the answer
- **User prompt** — the actual email to work on

In [ ]:
system_prompt = """
You are an email assistant for a busy professional.

Given an email, you must:
- Write a concise summary (3–6 bullets): purpose, asks, deadlines, decisions needed
- Suggest one short subject line (max 8 words), specific and scannable
- Call out urgency if a deadline or meeting is mentioned

Respond in markdown with exactly these two sections:

## Subject
<one line, no quotes>

## Summary
- bullet
- bullet

Do not wrap the markdown in a code block — respond with the markdown only.
"""

In [ ]:
def user_prompt_for(email_text: str) -> str:
    return f"""
Please summarize this email and suggest a subject line.

Email:
{email_text}
"""

## Messages

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```

In [ ]:
def messages_for(email_text: str):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(email_text)},
    ]

messages_for(email)

## Call Chat Completions

In [ ]:
def summarize_email(email_text: str) -> str:
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages_for(email_text),
    )
    return response.choices[0].message.content

result = summarize_email(email)
display(Markdown(result))

## Try another email

In [ ]:
another_email = """
From: Manish Kumar <manish@vendor.io>
To: Hitesh Kumar <hitesh@acme.com>
Subject: Invoice 1842

Hi Hitesh,

Invoice 1842 for the August GPU hours ($1,240) is attached. Payment is due
in 14 days. If you need the usage breakdown split by project, reply and I
will send it.

Alex
"""

display(Markdown(summarize_email(another_email)))